Single run analysis

In [ ]:
from pathlib import Path
from datetime import datetime
import sys

# Resolve repository root regardless of where notebook is executed from
repo_root = Path.cwd().parent if Path.cwd().name == "Postprocessing" else Path.cwd()
postproc_dir = repo_root / "Postprocessing"
if str(postproc_dir) not in sys.path:
    sys.path.insert(0, str(postproc_dir))

# Use latest BL KM run outputs automatically
output_bl = repo_root / "DELFTBLUE_SET" / "output" / "BL"
nc_files = sorted(output_bl.glob("BL_KMbase_run_KM_*.nc"), key=lambda p: p.stat().st_mtime)
yaml_files = sorted(output_bl.glob("BL_KMbase_run_KM_*.yaml"), key=lambda p: p.stat().st_mtime)
if not nc_files or not yaml_files:
    raise FileNotFoundError(f"No BL_KM output files found in {output_bl}")

netcdf_filename = str(nc_files[-1])
yaml_filename = str(yaml_files[-1])

stamp = datetime.now().strftime("%Y%m%d_%H%M")
analysis_dir = repo_root / "analysis_results"
analysis_dir.mkdir(parents=True, exist_ok=True)

flow_csv_filename = str(analysis_dir / "flow_with_utilization_and_coords.csv")
analysis_html_filename = str(analysis_dir / f"flows_costs_util_{stamp}.html")
installed_html_filename = str(analysis_dir / f"Installed_Capacities_{stamp}.html")
map_output_html = str(analysis_dir / f"carrier_flows_{stamp}.html")
geojson_filename = str(repo_root / "Postprocessing" / "NUTS2.geojson")
html_title = "Installed Capacities"
map_title = "Carrier flows - All Timesteps"

print(f"Using NC:   {netcdf_filename}")
print(f"Using YAML: {yaml_filename}")

from postprocessing import run_IC
from flows_cost_util import generate_report
from map import run_energy_flow_visualization

run_IC(
    netcdf_path=netcdf_filename,
    model_yaml_path=yaml_filename,
    output_html_path=installed_html_filename,
    html_title=html_title
)

generate_report(
    nc_file_path=netcdf_filename,
    processed_csv_path=flow_csv_filename,
    output_html_file=analysis_html_filename
)

run_energy_flow_visualization(
    title=map_title,
    netcdf_path=netcdf_filename,
    flow_csv_path=flow_csv_filename,
    yaml_path=yaml_filename,
    geojson_path=geojson_filename,
    output_html_path=map_output_html
)

print("Generated files:")
print(f" - {installed_html_filename}")
print(f" - {analysis_html_filename}")
print(f" - {map_output_html}")


Comparison tables

In [3]:
# Jupyter Cell Script for Calliope Model Comparison
# Run this cell to load the comparison functions and use them

import sys
from pathlib import Path

# Add the script path to Python path
repo_root = Path.cwd().parent if Path.cwd().name == "Postprocessing" else Path.cwd()
script_path = repo_root / "Postprocessing"
if str(script_path) not in sys.path:
    sys.path.append(str(script_path))

# Import the comparison functions
from calliope_comparison_script import compare_two_runs, detailed_capacity_comparison

def quick_compare(netcdf_1, yaml_1, netcdf_2, yaml_2, run_1_name="Run 1", run_2_name="Run 2", 
                  save_csv=True):
    """Quick comparison function for Jupyter notebook use"""
    
    print(f"ðŸ”„ Comparing {run_1_name} vs {run_2_name}")
    print("="*60)
    
    # Run the comparison
    comparison_df = compare_two_runs(
        netcdf_1, yaml_1, netcdf_2, yaml_2,
        run_1_name=run_1_name, run_2_name=run_2_name
    )
    
    # Get detailed capacity comparison
    detailed_df = detailed_capacity_comparison(
        netcdf_1, yaml_1, netcdf_2, yaml_2, 
        run_1_name=run_1_name, run_2_name=run_2_name
    )
    
    if save_csv:
        # Save to analysis_results folder
        output_dir = Path.cwd() / "analysis_results"
        output_dir.mkdir(parents=True, exist_ok=True)
        
        comparison_file = output_dir / f"comparison_{run_1_name}_{run_2_name}.csv"
        detailed_file = output_dir / f"detailed_capacity_{run_1_name}_{run_2_name}.csv"
        
        comparison_df.to_csv(comparison_file, index=False)
        detailed_df.to_csv(detailed_file, index=False)
        
        print(f"\nðŸ“ Results saved to:")
        print(f"   - {comparison_file}")
        print(f"   - {detailed_file}")
    
    return comparison_df, detailed_df

# Convenience functions
def compare_baseline_vs_smr(baseline_nc, baseline_yaml, smr_nc, smr_yaml):
    """Convenience function for comparing baseline vs SMR scenarios"""
    return quick_compare(
        baseline_nc, baseline_yaml, smr_nc, smr_yaml,
        run_1_name="Baseline", run_2_name="SMR_Scenario", 
        save_csv=True
    )

def compare_smr_variations(smr1_nc, smr1_yaml, smr2_nc, smr2_yaml, 
                          variation1_name="SMR_Var1", variation2_name="SMR_Var2"):
    """Convenience function for comparing different SMR scenario variations"""
    return quick_compare(
        smr1_nc, smr1_yaml, smr2_nc, smr2_yaml,
        run_1_name=variation1_name, run_2_name=variation2_name,
        save_csv=True
    )

print("âœ… Calliope comparison functions loaded successfully!")


âœ… Calliope comparison functions loaded successfully!


In [ ]:
# Comparison example disabled for automated postprocessing runs.
# Use Post_processing_comparison.ipynb for scenario-to-scenario comparisons.
print('Comparison example skipped; run Post_processing_comparison.ipynb if needed.')
